# Notebook 1: Data Preprocessing (AKT1 Bioactivity)


# Environment Setup

In [ ]:
!pip -q install rdkit optuna xgboost shap


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import rdkit
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Descriptors, inchi, rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize

from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import matthews_corrcoef, r2_score
import sklearn

import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
RDLogger.DisableLog("rdApp.*")

print("Package versions (record these for reproducibility -- RDKit's 2D "
      "descriptor list can change between versions, which changes the "
      "feature columns produced by Descriptors._descList):")
print(f"  RDKit: {rdkit.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "font.size": 10,
})

ACTIVE_PIC50_CUTOFF = 7.0
INACTIVE_PIC50_CUTOFF = 6.0

TIER_TO_LABEL = {"Inactive": 0, "Intermediate": 1, "Active": 2}
LABEL_TO_TIER = {v: k for k, v in TIER_TO_LABEL.items()}
CLASS_LABELS_ORDERED = [0, 1, 2]
TIER_NAMES_ORDERED = ["Inactive", "Intermediate", "Active"]

AD_TANIMOTO_THRESHOLD = 0.50
AD_KNN_K = 5
# %% [code] cell 4
csv_path = "AKT1_clean.csv"

try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
    df = pd.read_csv(csv_path)

SMILES_COL = "canonical_smiles"
PIC50_COL = "pIC50"

df[PIC50_COL] = pd.to_numeric(df[PIC50_COL], errors="coerce")
df = df.dropna(subset=[SMILES_COL, PIC50_COL])
df = df.drop_duplicates(subset=[SMILES_COL]).reset_index(drop=True)

print(f"Loaded {df.shape[0]} unique molecules from {csv_path}.")

## Manuscript Figure: Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["pIC50"], bins=30, color="#4C78A8", edgecolor="white")
axes[0].axvline(ACTIVE_PIC50_CUTOFF, color="#B23A48", linestyle="--", linewidth=1, label=f"Active >= {ACTIVE_PIC50_CUTOFF}")
axes[0].axvline(INACTIVE_PIC50_CUTOFF, color="0.4", linestyle="--", linewidth=1, label=f"Inactive < {INACTIVE_PIC50_CUTOFF}")
axes[0].set_xlabel("pIC50")
axes[0].set_ylabel("Count")
axes[0].set_title("pIC50 distribution")
axes[0].legend(frameon=False, fontsize=8)

axes[1].scatter(df["MW"], df["LogP"], s=14, alpha=0.5, color="#4C78A8")
axes[1].set_xlabel("Molecular weight")
axes[1].set_ylabel("LogP")
axes[1].set_title("Chemical space (MW vs. LogP)")

plt.tight_layout()
plt.savefig("figure_01_dataset_overview.png", bbox_inches="tight")
plt.show()


# Activity Labeling: Three-Tier Bioactivity

In [ ]:
def assign_bioactivity_tier(pic50):
    if pic50 >= ACTIVE_PIC50_CUTOFF:
        return "Active"
    elif pic50 >= INACTIVE_PIC50_CUTOFF:
        return "Intermediate"
    return "Inactive"


df["bioactivity_tier"] = df["pIC50"].apply(assign_bioactivity_tier)
y_class_full = np.array([TIER_TO_LABEL[t] for t in df["bioactivity_tier"]])

tier_counts = df["bioactivity_tier"].value_counts().reindex(TIER_NAMES_ORDERED[::-1]).fillna(0).astype(int)
print("Bioactivity tier counts:")
print(tier_counts)


# Molecular Featurization: ECFP4 + RDKit Topological + MACCS


In [ ]:
_ecfp4_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
_ecfp6_generator = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
_fcfp4_invariant_gen = rdFingerprintGenerator.GetMorganFeatureAtomInvGen()
_fcfp4_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, atomInvariantsGenerator=_fcfp4_invariant_gen
)


def morgan_fp(mol, generator=_ecfp4_generator, n_bits=2048):
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def morgan_fp_ecfp6(mol, n_bits=2048):
    return morgan_fp(mol, generator=_ecfp6_generator, n_bits=n_bits)


def morgan_fp_fcfp4(mol, n_bits=2048):
    return morgan_fp(mol, generator=_fcfp4_generator, n_bits=n_bits)


def rdkit_topological_fp(mol, n_bits=2048):
    fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def maccs_fp(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((167,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def hybrid_ecfp4_rdkit_maccs(mol):
    return np.concatenate([morgan_fp(mol), rdkit_topological_fp(mol), maccs_fp(mol)])


def to_mol(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        return mol  # Returns None if SMILES is invalid
    except Exception:
        return None

df['mol'] = df[SMILES_COL].apply(to_mol)
before_mol_validation = len(df)
df = df[df['mol'].notna()].reset_index(drop=True)
if len(df) < before_mol_validation:
    print(f"Dropped {before_mol_validation - len(df)} rows with invalid SMILES after RDKit mol conversion.")

mols = df["mol"].tolist()

descriptor_names = [name for name, func in Descriptors._descList]
descriptor_funcs = [func for name, func in Descriptors._descList]


def calc_2d_descriptors(mol):
    values = []
    for func in descriptor_funcs:
        try:
            values.append(func(mol))
        except Exception:
            values.append(np.nan)
    return values


ecfp4_all = np.array([morgan_fp(m) for m in mols])
rdkit_topo_all = np.array([rdkit_topological_fp(m) for m in mols])
maccs_all = np.array([maccs_fp(m) for m in mols])
hybrid_all = np.concatenate([ecfp4_all, rdkit_topo_all, maccs_all], axis=1)

desc_all = np.array([calc_2d_descriptors(m) for m in mols])
desc_cols = [f"DESC_{name}" for name in descriptor_names]
desc_df_all = pd.DataFrame(desc_all, columns=desc_cols).replace([np.inf, -np.inf], np.nan)


y_pic50_all = df["pIC50"].values

print("ECFP4:", ecfp4_all.shape, "| RDKit topological:", rdkit_topo_all.shape,
      "| MACCS:", maccs_all.shape, "| Hybrid:", hybrid_all.shape,
      "| 2D descriptors:", desc_df_all.shape)


def nearest_training_tanimoto(query_fps, train_fps, batch_size=500):
    query_bool = np.asarray(query_fps).astype(np.int32)
    train_bool = np.asarray(train_fps).astype(np.int32)
    train_popcount = train_bool.sum(axis=1)
    nearest = np.empty(len(query_bool), dtype=float)
    for start in range(0, len(query_bool), batch_size):
        batch = query_bool[start:start + batch_size]
        query_popcount = batch.sum(axis=1)
        intersection = batch @ train_bool.T
        union = query_popcount[:, None] + train_popcount[None, :] - intersection
        similarity = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
        nearest[start:start + batch_size] = similarity.max(axis=1)
    return nearest


def topk_mean_training_tanimoto(query_fps, train_fps, k=5, batch_size=500):
    query_bool = np.asarray(query_fps).astype(np.int32)
    train_bool = np.asarray(train_fps).astype(np.int32)
    train_popcount = train_bool.sum(axis=1)
    k_eff = min(k, len(train_bool))
    topk_mean = np.empty(len(query_bool), dtype=float)
    for start in range(0, len(query_bool), batch_size):
        batch = query_bool[start:start + batch_size]
        query_popcount = batch.sum(axis=1)
        intersection = batch @ train_bool.T
        union = query_popcount[:, None] + train_popcount[None, :] - intersection
        similarity = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
        top_k_vals = -np.partition(-similarity, k_eff - 1, axis=1)[:, :k_eff]
        topk_mean[start:start + batch_size] = top_k_vals.mean(axis=1)
    return topk_mean

# Bemis-Murcko Scaffold Split: Train 80%, Validation 10%, Test 10%

In [ ]:
def get_scaffold(mol):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None


df["scaffold"] = [get_scaffold(m) for m in mols]


def scaffold_split(data, scaffold_col="scaffold", train_frac=0.8, val_frac=0.1, test_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    scaffold_groups = data.groupby(scaffold_col).indices
    scaffold_sets = [np.array(idx) for idx in scaffold_groups.values()]
    # Shuffle before stable sorting so different seeds change the placement
    # of equal-size scaffold groups.
    rng.shuffle(scaffold_sets)
    scaffold_sets = sorted(scaffold_sets, key=len, reverse=True)

    n_total = len(data)
    train_cutoff = train_frac * n_total
    val_cutoff = (train_frac + val_frac) * n_total
    train_idx, val_idx, test_idx = [], [], []
    for s in scaffold_sets:
        if len(train_idx) + len(s) <= train_cutoff:
            train_idx.extend(s)
        elif len(train_idx) + len(val_idx) + len(s) <= val_cutoff:
            val_idx.extend(s)
        else:
            test_idx.extend(s)
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


train_idx, val_idx, test_idx = scaffold_split(df, seed=RANDOM_STATE)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))


## Manuscript Figure: Scaffold Split Summary

In [ ]:
split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Compounds": [len(train_idx), len(val_idx), len(test_idx)],
    "Unique scaffolds": [
        df.iloc[train_idx]["scaffold"].nunique(),
        df.iloc[val_idx]["scaffold"].nunique(),
        df.iloc[test_idx]["scaffold"].nunique(),
    ],
})

fig, ax = plt.subplots(figsize=(6, 4))
width = 0.38
x = np.arange(len(split_summary))
ax.bar(x - width / 2, split_summary["Compounds"], width, label="Compounds", color="#4C78A8")
ax.bar(x + width / 2, split_summary["Unique scaffolds"], width, label="Unique scaffolds", color="#F58518")
ax.set_xticks(x)
ax.set_xticklabels(split_summary["Split"])
ax.set_ylabel("Count")
ax.set_title("Bemis-Murcko scaffold split")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("figure_02_scaffold_split.png", bbox_inches="tight")
plt.show()

split_summary


# Leakage-Safe Feature Filtering


In [ ]:
def prepare_filtered_split(train_idx_s, val_idx_s, test_idx_s, fp_source, desc_source,
                            variance_threshold=0.01, corr_threshold=0.95):
    fp_cols_s = [f"FP_{i}" for i in range(fp_source.shape[1])]
    X_fp = pd.DataFrame(fp_source, columns=fp_cols_s)
    X_all_s = pd.concat([X_fp, desc_source.reset_index(drop=True)], axis=1)

    X_train_raw = X_all_s.iloc[train_idx_s].copy()
    X_val_raw = X_all_s.iloc[val_idx_s].copy()
    X_test_raw = X_all_s.iloc[test_idx_s].copy()

    medians = X_train_raw[desc_cols].median(numeric_only=True).fillna(0.0)
    X_train_raw[desc_cols] = X_train_raw[desc_cols].fillna(medians)
    X_val_raw[desc_cols] = X_val_raw[desc_cols].fillna(medians)
    X_test_raw[desc_cols] = X_test_raw[desc_cols].fillna(medians)

    selector = VarianceThreshold(threshold=variance_threshold)
    X_train_var = pd.DataFrame(
        selector.fit_transform(X_train_raw), columns=X_train_raw.columns[selector.get_support()]
    )
    X_val_var = pd.DataFrame(selector.transform(X_val_raw), columns=X_train_var.columns)
    X_test_var = pd.DataFrame(selector.transform(X_test_raw), columns=X_train_var.columns)

    corr_matrix = X_train_var.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > corr_threshold)]

    return (
        X_train_var.drop(columns=to_drop), X_val_var.drop(columns=to_drop), X_test_var.drop(columns=to_drop),
        medians, selector, to_drop,
    )


# Fingerprint Ablation: Is the ECFP4 + RDKit + MACCS Hybrid the Right Choice?

In [ ]:
FINGERPRINT_VARIANTS = {
    "ECFP4 (radius 2, 2048 bit)": ecfp4_all,
    "ECFP6 (radius 3, 2048 bit)": np.array([morgan_fp_ecfp6(m) for m in mols]),
    "FCFP4 (radius 2, 2048 bit, feature-based)": np.array([morgan_fp_fcfp4(m) for m in mols]),
    "MACCS keys (167 bit)": maccs_all,
    "RDKit topological (2048 bit)": rdkit_topo_all,
    "Hybrid ECFP4 + RDKit + MACCS (4263 bit)": hybrid_all,
}

ablation_rows = []
for fp_name, fp_arr in FINGERPRINT_VARIANTS.items():
    print(f"Evaluating fingerprint: {fp_name}")
    X_tr, X_va, X_te, *_ = prepare_filtered_split(train_idx, val_idx, test_idx, fp_arr, desc_df_all)

    rf_clf_quick = RandomForestClassifier(
        n_estimators=400, max_depth=20, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_clf_quick.fit(X_tr, y_class_full[train_idx])
    mcc_v = matthews_corrcoef(y_class_full[val_idx], rf_clf_quick.predict(X_va))

    rf_reg_quick = RandomForestRegressor(n_estimators=400, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1)
    rf_reg_quick.fit(X_tr, y_pic50_all[train_idx])
    r2_v = r2_score(y_pic50_all[val_idx], rf_reg_quick.predict(X_va))

    ablation_rows.append({
        "Fingerprint": fp_name,
        "Raw features (fp + 2D desc)": fp_arr.shape[1] + desc_df_all.shape[1],
        "Features after leakage-safe filtering": X_tr.shape[1],
        "Validation MCC (quick RF classifier)": mcc_v,
        "Validation R2 (quick RF regressor)": r2_v,
    })

fingerprint_ablation_df = pd.DataFrame(ablation_rows).sort_values(
    "Validation R2 (quick RF regressor)", ascending=False
).reset_index(drop=True)

print("\nFingerprint ablation (same scaffold split, untuned RF, screening only):")
fingerprint_ablation_df


## Committing to the Hybrid Fingerprint


# Final Feature Matrix: Leakage-Safe Filtered Hybrid Features

In [ ]:
X_train, X_val, X_test, descriptor_medians, var_selector, corr_dropped_columns = prepare_filtered_split(
    train_idx, val_idx, test_idx, hybrid_all, desc_df_all
)

y_train_pic50 = y_pic50_all[train_idx]
y_val_pic50 = y_pic50_all[val_idx]
y_test_pic50 = y_pic50_all[test_idx]

y_train_class = y_class_full[train_idx]
y_val_class = y_class_full[val_idx]
y_test_class = y_class_full[test_idx]

print("Final feature matrix (hybrid fingerprint + 2D descriptors, leakage-safe filtered):")
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

print("\nClass balance (0=Inactive, 1=Intermediate, 2=Active):")
for split_name, y_split in [("Train", y_train_class), ("Validation", y_val_class), ("Test", y_test_class)]:
    counts = pd.Series(y_split).value_counts().reindex(CLASS_LABELS_ORDERED).fillna(0).astype(int)
    counts.index = TIER_NAMES_ORDERED
    print(f"  {split_name}: {counts.to_dict()}")


# Repeated Scaffold-Split Seeds


In [ ]:
REPEATED_SPLIT_SEEDS = [11, 22, 33, 44, 55]
repeated_splits = {seed: scaffold_split(df, seed=seed) for seed in REPEATED_SPLIT_SEEDS}

for seed, (tr, va, te) in repeated_splits.items():
    print(f"  seed {seed}: train={len(tr)} val={len(va)} test={len(te)}")


# Save Everything: `features.pkl` and `splits.pkl`

In [ ]:
features_payload = {
    "smiles": df["canonical_smiles"].tolist(),
    "pIC50": y_pic50_all,
    "bioactivity_tier": df["bioactivity_tier"].values,
    "y_class": y_class_full,
    "scaffold": df["scaffold"].values,
    "inchikey": df["inchikey"].values,
    "ecfp4": ecfp4_all,
    "rdkit_topo": rdkit_topo_all,
    "maccs": maccs_all,
    "hybrid_fp": hybrid_all,
    "desc_2d": desc_df_all,
    "desc_cols": desc_cols,
    "descriptor_names": descriptor_names,
    "fingerprint_ablation": fingerprint_ablation_df,
    "constants": {
        "RANDOM_STATE": RANDOM_STATE,
        "ACTIVE_PIC50_CUTOFF": ACTIVE_PIC50_CUTOFF,
        "INACTIVE_PIC50_CUTOFF": INACTIVE_PIC50_CUTOFF,
        "TIER_TO_LABEL": TIER_TO_LABEL,
        "LABEL_TO_TIER": LABEL_TO_TIER,
        "CLASS_LABELS_ORDERED": CLASS_LABELS_ORDERED,
        "TIER_NAMES_ORDERED": TIER_NAMES_ORDERED,
        "AD_TANIMOTO_THRESHOLD": AD_TANIMOTO_THRESHOLD,
        "AD_KNN_K": AD_KNN_K,
        "rdkit_version": rdkit.__version__,
    },
}
joblib.dump(features_payload, "features.pkl")

splits_payload = {
    "train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx,
    "repeated_split_seeds": REPEATED_SPLIT_SEEDS,
    "repeated_splits": repeated_splits,
    "X_train": X_train, "X_val": X_val, "X_test": X_test,
    "y_train_pic50": y_train_pic50, "y_val_pic50": y_val_pic50, "y_test_pic50": y_test_pic50,
    "y_train_class": y_train_class, "y_val_class": y_val_class, "y_test_class": y_test_class,
    "feature_columns": list(X_train.columns),
    "descriptor_medians": descriptor_medians,
    "correlation_dropped_columns": corr_dropped_columns,
}
joblib.dump(splits_payload, "splits.pkl")

print("Saved features.pkl and splits.pkl")
print(f"  features.pkl: {len(features_payload['smiles'])} molecules")
print(f"  splits.pkl: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}, "
      f"{X_train.shape[1]} final feature columns")

try:
    from google.colab import files
    files.download("features.pkl")
    files.download("splits.pkl")
except Exception:
    print("Not running in Colab (or download failed) -- retrieve features.pkl / "
          "splits.pkl manually from the working directory.")
